# Periodic curves: dense MLP, shared MLP and CNN

This notebook accompanies Exercise 2.5 of the course. We want to reconstruct a family of periodic curves from amplitude, frequency and shift.

The experiment compares two machines. The first produces the entire curve using dense matrices. The second applies the same MLP at every grid point: this structure corresponds exactly to a one-dimensional CNN whose kernels have width $1$.

## Outline

1. [Constructing the data](#donnees-periodiques)
2. [Dense MLP](#mlp-dense)
3. [Shared MLP](#mlp-partage)
4. [Training and comparison](#comparaison-machines)
5. [Weight sharing and equivariance](#equivariance)
6. [Extension: periodic convolution](#convolution-periodique)

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import matplotlib.pyplot as plt
import optax
import flax
from flax import nnx

print(f"JAX {jax.__version__}, Flax {flax.__version__}, Optax {optax.__version__}")

<a id="donnees-periodiques"></a>
## 1. Constructing the data

On the periodic grid $x_j=j/n_x$, consider
$$
u_{a,k,\tau}(x)=a\sin\bigl(2\pi k(x-\tau)\bigr),
\qquad
U(a,k,\tau)=\bigl(u_{a,k,\tau}(x_j)\bigr)_j.
$$

Training shifts are even multiples of $1/16$; test shifts are odd multiples. Testing therefore uses shifts absent from the training data.

In [ ]:
n_x = 64
k_max = 5
grille = jnp.arange(n_x) / n_x

def courbes(parametres):
    a = parametres[:, 0:1]
    k = parametres[:, 1:2]
    tau = parametres[:, 2:3]
    return a * jnp.sin(2.0 * jnp.pi * k * (grille[None, :] - tau))

def generer_parametres(cle, n_donnees, indices_tau):
    cle_a, cle_k, cle_tau = jax.random.split(cle, 3)
    a = jax.random.uniform(cle_a, (n_donnees,), minval=0.5, maxval=1.5)
    k = jax.random.randint(cle_k, (n_donnees,), minval=1, maxval=k_max + 1)
    indice_tau = jax.random.choice(cle_tau, indices_tau, shape=(n_donnees,))
    tau = indice_tau / 16.0
    return jnp.stack((a, k, tau), axis=1)

cle_apprentissage, cle_test = jax.random.split(jax.random.key(2026))
parametres_apprentissage = generer_parametres(
    cle_apprentissage, 512, jnp.arange(0, 16, 2)
)
parametres_test = generer_parametres(
    cle_test, 256, jnp.arange(1, 16, 2)
)
z_apprentissage = courbes(parametres_apprentissage)
z_test = courbes(parametres_test)

print(parametres_apprentissage.shape, z_apprentissage.shape)

In [ ]:
plt.figure(figsize=(7, 3.5))
for i in range(6):
    a, k, tau = parametres_apprentissage[i]
    plt.plot(grille, z_apprentissage[i], label=f"a={a:.2f}, k={int(k)}, τ={tau:.2f}")
plt.xlabel("x")
plt.grid(True)
plt.legend(ncol=2, fontsize=8)
plt.show()

<a id="mlp-dense"></a>
## 2. Dense MLP

Use Flax NNX to construct a dense MLP
$$
G_p:\mathbb R^3\longrightarrow\mathbb R^{n_x}.
$$

Use two hidden layers of width $64$ and the `tanh` activation. The final-layer matrices are dense: each output value therefore has its own coefficients.

In [ ]:
# Complete this section.

<a id="mlp-partage"></a>
## 3. Shared MLP

Associate $(a,k,\tau)$ with the spatial input $X(a,k,\tau)\in\mathbb R^{n_x\times4}$ given by
$$
X(a,k,\tau)_{j,:}
=\bigl(a,k,\cos(2\pi(x_j-\tau)),\sin(2\pi(x_j-\tau))\bigr).
$$

Then construct an MLP $g_q:\mathbb R^4\to\mathbb R$ and apply the **same** MLP to each row:
$$
\bigl(C_q(X)\bigr)_j=g_q(X_{j,:}).
$$

With NNX, a `Linear` layer automatically acts on the last array dimension: preceding dimensions index data and positions.

In [ ]:
def entree_spatiale(parametres):
    a = parametres[:, 0:1]
    k = parametres[:, 1:2]
    tau = parametres[:, 2:3]
    phase = 2.0 * jnp.pi * (grille[None, :] - tau)
    return jnp.stack(
        (
            jnp.broadcast_to(a, phase.shape),
            jnp.broadcast_to(k, phase.shape),
            jnp.cos(phase),
            jnp.sin(phase),
        ),
        axis=-1,
    )

x_partage_apprentissage = entree_spatiale(parametres_apprentissage)
x_partage_test = entree_spatiale(parametres_test)
print(x_partage_apprentissage.shape)

In [ ]:
# Complete this section.

<a id="comparaison-machines"></a>
## 4. Training and comparison

The data construction and training loop are provided. Fit both machines using the same mean squared error (MSE), then compare their errors on shifts absent from the training set.

In [ ]:
def cout(machine, x, z):
    return jnp.mean((machine(x) - z) ** 2)

@nnx.jit
def pas_apprentissage(machine, optimiseur, x, z):
    valeur, gradient = nnx.value_and_grad(cout)(machine, x, z)
    optimiseur.update(machine, gradient)
    return valeur

def apprendre(machine, x, z, n_iterations=1500, alpha=3.0e-3):
    optimiseur = nnx.Optimizer(machine, optax.adam(alpha), wrt=nnx.Param)
    historique = []
    for _ in range(n_iterations):
        valeur = pas_apprentissage(machine, optimiseur, x, z)
        historique.append(float(valeur))
    return historique

In [ ]:
# Complete this section.

In [ ]:
# Complete this section.

### Number of parameters

1. How many parameters does a dense layer with $r$ inputs and $n_x$ outputs have?
2. How many does a shared layer with $c_{\rm in}$ input channels and $c_{\rm out}$ output channels have?
3. Verify the total number of parameters in both machines constructed above.

In [ ]:
# Complete this section.

<a id="equivariance"></a>
## 5. Weight sharing and equivariance

For $r\in\mathbb Z$, let
$$
(T_rv)_j=v_{(j-r)\bmod n_x}.
$$

1. Verify the identities
   $$
   T_rU(a,k,\tau)=U\left(a,k,\tau+\frac r{n_x}\right),
   \qquad
   T_rX(a,k,\tau)=X\left(a,k,\tau+\frac r{n_x}\right).
   $$
2. Verify that $C_q(T_rX)=T_rC_q(X)$, even before training.
3. Explain why the shared MLP is exactly a one-dimensional CNN with kernel width $1$.

In [ ]:
# Complete this section.

<a id="convolution-periodique"></a>
## 6. Extension: periodic convolution

For a kernel of width $2s+1$, the following function directly implements periodic convolution. The last dimension contains channels; the penultimate dimension is the spatial index.

Use this function to verify equivariance for a layer of width $3$ or $5$. Then optionally replace the shared MLP's `Linear` layers with layers of this type and learn the kernels.

In [ ]:
def convolution_periodique(x, noyau, biais):
    largeur = noyau.shape[0]
    assert largeur % 2 == 1
    s = largeur // 2
    y = jnp.broadcast_to(biais, x.shape[:-1] + (biais.size,))
    for indice, decalage in enumerate(range(-s, s + 1)):
        y = y + jnp.roll(x, decalage, axis=-2) @ noyau[indice]
    return y

In [ ]:
# Complete this section.

## Summary

The dense MLP and shared MLP can both approximate the curves. The essential difference is structural: the shared MLP uses the same parameters at every position, has a number of parameters per layer independent of $n_x$, and is translation equivariant.

A CNN of width $1$ is therefore not a new kind of composition: it is an MLP applied at each point with weight sharing. Wider kernels add local interactions between neighbouring positions.